# Pandas Playground — Manipulación de columnas

Notebook de práctica para aprender a crear columnas, aplicar condiciones, y manipular strings.
Usamos los datos reales de `ovilos-prod.xlsx` como ejemplo.

**Objetivo:** entender los patrones básicos para después aplicarlos en el notebook de limpieza.

In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

SRC = Path("../../reference/warehouse/cleaned/ovillos-prod.xlsx")
df = pd.read_excel(SRC)
print(f"Shape: {df.shape}")
print(f"Columnas: {list(df.columns)}")

---
## 1. Crear columnas nuevas

In [ ]:
# 1.1 Columna con valor fijo (se repite en todas las filas)
df['origen'] = 'sic-jac'
df[['codigo', 'origen']].head(5)

In [ ]:
# 1.2 Columna copiando / transformando otra
df['codigo_str'] = df['codigo'].astype(str)
df['descripcion_upper'] = df['descripcion'].str.upper()
df[['codigo', 'codigo_str', 'descripcion', 'descripcion_upper']].head(3)

In [ ]:
# 1.3 Columna booleana (True/False) a partir de una condición
df['es_recuperado'] = df['codigo'].astype(str).str.startswith('REC-')
df['es_sm'] = df['codigo'].astype(str).str.endswith('-SM')
df[['codigo', 'es_recuperado', 'es_sm']].head(10)

In [ ]:
# 1.4 Filtrar solo donde la condición se cumple
print("Códigos que empiezan con REC-:")
print(df[df['es_recuperado']][['codigo', 'descripcion']].to_string(index=False))

---
## 2. Llenar columnas condicionalmente (.loc)

In [ ]:
# 2.1 Crear columna vacía y llenar SÓLO donde se cumple condición
df['tipo_material'] = None  # toda la columna arranca en None

df.loc[df['codigo'].astype(str).str.startswith('REC-'), 'tipo_material'] = 'REC'
df.loc[df['codigo'].astype(str).str.startswith('MAT-'), 'tipo_material'] = 'MAT-REC'
df.loc[df['codigo'].astype(str).str.endswith('-SM'), 'tipo_material'] = 'SM'
df.loc[df['codigo'].astype(str).str.endswith('-N'), 'tipo_material'] = 'N'

print("Distribución:")
print(df['tipo_material'].value_counts(dropna=False).to_string())
print()
print("Ejemplos:")
df[df['tipo_material'].notna()][['codigo', 'tipo_material']].head(10)

In [ ]:
# 2.2 Cuidado: el orden importa cuando hay valores que cumplen MÚLTIPLES condiciones
# Ejemplo: REC-ALM-1-N cumple startswith('REC-') Y endswith('-N')
# La última asignación gana

print("REC-ALM-1-N:")
print(df[df['codigo'] == 'REC-ALM-1-N'][['codigo', 'tipo_material']])

# Solución: asignar primero lo más general, después lo más específico
# O asegurarse de que las condiciones no se solapen

---
## 3. Múltiples condiciones con np.select (más limpio)

In [ ]:
# np.select(lista_de_condiciones, lista_de_valores, default=None)
# Evalúa en orden: PRIMERA condición que se cumple → ese valor

conditions = [
    df['codigo'].astype(str).str.startswith('REC-'),
    df['codigo'].astype(str).str.startswith('MAT-'),
    df['codigo'].astype(str).str.endswith('-SM'),
    df['codigo'].astype(str).str.endswith('-N'),
]
choices = ['REC', 'MAT-REC', 'SM', 'N']

df['tipo_v2'] = np.select(conditions, choices, default='estandar')

print("Distribución np.select:")
print(df['tipo_v2'].value_counts().to_string())
print()
df[['codigo', 'tipo_v2']].head(10)

In [ ]:
# También podés usar np.where anidado (menos legible con muchas condiciones)

df['tipo_v3'] = np.where(
    df['codigo'].astype(str).str.startswith('REC-'), 'REC',
    np.where(
        df['codigo'].astype(str).str.startswith('MAT-'), 'MAT-REC',
        np.where(
            df['codigo'].astype(str).str.endswith('-SM'), 'SM',
            'estandar'
        )
    )
)

print("np.where anidado:")
print(df['tipo_v3'].value_counts().to_string())

---
## 4. Extraer y eliminar partes de strings

In [ ]:
# 4.1 str.extract() — sacar una parte usando regex
# Busca el patrón y devuelve SÓLO lo que está entre paréntesis (captura)

print("Sufijo alfabético al final del código:")
sufijos = df['codigo'].astype(str).str.extract(r'-(SM|SN|CH|N|M)$')
print(sufijos[0].value_counts(dropna=False).to_string())
print()

print("Prefijo (primeros caracteres hasta el primer -):")
prefijos = df['codigo'].astype(str).str.extract(r'^([A-Za-z]+)-')
print(prefijos[0].value_counts(dropna=False).head(10).to_string())

In [ ]:
# 4.2 str.replace() — eliminar o reemplazar partes

# Eliminar sufijos específicos del código
df['codigo_limpio'] = df['codigo'].astype(str).str.replace(
    r'-(SM|SN|CH|N|M)$',  # patrón: guión + letras al final
    '',                     # reemplazar con vacío (eliminar)
    regex=True
)

print("Eliminar sufijos:")
ejemplos = df[df['codigo'] != df['codigo_limpio']][['codigo', 'codigo_limpio']].drop_duplicates()
print(ejemplos.to_string(index=False))

In [ ]:
# 4.3 str.replace() — reemplazar texto específico

# Ejemplo: normalizar nombres de colores en descripción
print("Original:", df['descripcion'].iloc[500])
print()

# Reemplazar texto exacto
test = df['descripcion'].iloc[500].replace('CICLAN', 'CYCLAN')
print(f"Reemplazo simple: {test}")

In [ ]:
# 4.4 str.split() — separar en columnas

# Split por doble espacio
partes = df['descripcion'].str.split(r'\s{2,}', expand=True)
print(f"Partes por doble espacio: {partes.shape[1]} columnas máx")
print()

# Split del código por guión
code_parts = df['codigo'].astype(str).str.split('-', expand=True)
print(f"Partes del código: {code_parts.shape[1]} columnas")
display(code_parts.head(10))

In [ ]:
# 4.5 Renombrar columnas del split
code_parts.columns = ['nivel1', 'nivel2', 'nivel3', 'nivel4', 'nivel5', 'nivel6']

# Agregar al dataframe original
df[['n1', 'n2', 'n3']] = df['codigo'].astype(str).str.split('-', expand=True).iloc[:, :3]
print("Código descompuesto:")
df[['codigo', 'n1', 'n2', 'n3']].head(10)

---
## 5. Funciones personalizadas con .apply()

In [ ]:
# 5.1 Función que clasifica según reglas complejas

def clasificar_material(codigo):
    """Clasifica un código en tipo de material según prefijos y sufijos."""
    if not isinstance(codigo, str):
        return 'desconocido'
    
    c = str(codigo)
    
    # Prefijos primero (más específicos primero)
    if c.startswith('MAT-REC'):
        return 'material_recuperado'
    if c.startswith('REC-'):
        return 'recuperado'
    if c.startswith('MAT-'):
        return 'material'
    
    # Sufijos
    if '-SM-' in c or c.endswith('-SM'):
        return 'semipeinado'
    if c.endswith('-SN'):
        return 'semipeinado_sn'
    if c.endswith('-CH'):
        return 'chanel'
    if c.endswith('-N'):
        return 'normal'
    if c.endswith('-M'):
        return 'madeja'
    if 'STOLL' in c:
        return 'stoll'
    
    return 'estandar'

df['clasificacion'] = df['codigo'].apply(clasificar_material)

print("Clasificación:")
print(df['clasificacion'].value_counts().to_string())
print()

# Ver algunos casos no estándar
print("Casos no estándar:")
no_std = df[df['clasificacion'] != 'estandar'][['codigo', 'descripcion', 'clasificacion']].drop_duplicates()
print(no_std.to_string(index=False))

In [ ]:
# 5.2 apply con lambda (para lógica simple)

df['desc_largo'] = df['descripcion'].apply(lambda x: len(str(x)))
df[['codigo', 'descripcion', 'desc_largo']].head(5)

---
## 6. Ejemplo práctico: construir tipo_material definitivo

In [ ]:
# Combinamos todo: condiciones ordenadas correctamente
# Reglas:
#   - REC-*          → 'REC'
#   - MAT-REC / MAT- → 'MAT-REC'
#   - *-SM           → 'SM'
#   - *-SN           → 'SN'
#   - *-CH           → 'CH'
#   - *-N            → 'N'
#   - lo demás       → None (para analizar después)

cod = df['codigo'].astype(str)

conditions = [
    cod.str.startswith('REC-'),
    cod.str.startswith('MAT-'),
    cod.str.endswith('-SM'),
    cod.str.endswith('-SN'),
    cod.str.endswith('-CH'),
    cod.str.endswith('-N'),
    cod.str.endswith('-M'),
]
choices = ['REC', 'MAT-REC', 'SM', 'SN', 'CH', 'N', 'M']

df['tipo_material_final'] = np.select(conditions, choices, default=None)

print("Distribución final:")
for v, c in df['tipo_material_final'].value_counts(dropna=False).items():
    label = str(v) if pd.notna(v) else 'None'
    print(f"  {label:>12}: {c:>4}")

print()
print("Ejemplos de cada tipo:")
for t in df['tipo_material_final'].dropna().unique():
    ej = df[df['tipo_material_final'] == t][['codigo']].iloc[0]
    print(f"  {t:>8} → {ej['codigo']}")

In [ ]:
# Bonus: mostrar los que quedaron sin clasificar
sin_tipo = df[df['tipo_material_final'].isna()][['codigo', 'descripcion']].drop_duplicates()
print(f"Códigos sin clasificar: {len(sin_tipo)}")
print()
print("Muestra:")
sin_tipo.head(20)

---
## Resumen: patrones que usamos

| Qué querés hacer | Código |
|---|---|
| Columna con valor fijo | `df['col'] = valor` |
| Columna desde otra | `df['col'] = df['otra'].str.upper()` |
| Columna booleana por condición | `df['col'] = df['x'].str.startswith('REC-')` |
| Llenar condicionalmente | `df.loc[condicion, 'col'] = valor` |
| Múltiples condiciones | `np.select([c1, c2], [v1, v2], default=x)` |
| Extraer con regex | `df['x'].str.extract(r'-(SM)$')` |
| Eliminar parte | `df['x'].str.replace(r'-SM$', '', regex=True)` |
| Separar en columnas | `df['x'].str.split('-', expand=True)` |
| Lógica compleja | `df['x'].apply(mi_funcion)` |